# E-Commerce Funnel & Customer Analytics

## Objectives:
1. Analyze user progression through the purchase funnel.
2. Identify which traffic sources drive the conversion rates.
3. Evaluate revenue drivers across departments, brands and customer segments.
4. Measure customer retention and repeat purchase behavior using cohort analysis.

### Dataset: 
Five interconnected tables - events, orders, order_items, users and products - covering user sessions, transactions and product metadata.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Loading events data
events=pd.read_csv('events.csv')
events.head()

In [ ]:
events.shape

In [ ]:
events.isnull().sum()

In [ ]:
events.event_type.value_counts()

In [ ]:
# Plotting Count by Event
event_counts=events.event_type.value_counts()
print(event_counts)
event_counts.plot(kind='bar')
plt.title('Count by Event Type')
plt.xlabel('Event Type')
plt.ylabel('Count of Event')
plt.show()

From the above graph, we infer that most customers are just looking at the product and adding it to cart rather than actually making a purchase.

In [ ]:
events.info()

In [ ]:
# Converting 'created_at' column to datetime datatype
events['created_at']=pd.to_datetime(events['created_at'], format='mixed')

In [ ]:
events.info()

In [ ]:
events.isnull().sum()

There are many null values in user_id column. This implies that nearly half the users have not logged in, meaning they might just be scrolling through the products.

In [ ]:
events.browser.value_counts()

In [ ]:
purchase_count_by_browser=events[events['event_type']=='purchase'].groupby('browser').size()
purchase_count_by_browser.plot(kind='bar')
plt.title('Purchase Count by Browser')
plt.xlabel('Browser')
plt.ylabel('Number of Purchases')
plt.show()

From the above graph, it is obvious that those who haved used the Chrome browser have made the most purchases, followed by Firefox and Safari users.

### Building Sequential Funnel

The funnel was constructed at session level using ordered event sequences. 
Each session's events were sorted by sequence_number to preserve chronological 
order. A session was counted at each stage only if the event occurred after 
the previous stage within the same session:

- **Viewed Product:** session contains a `product` event  
- **Added to Cart:** `cart` event occurred after product view  
- **Purchased:** `purchase` event occurred after cart addition

In [ ]:
# Keeping only relevant funnel events
funnel_events=events[events['event_type'].isin(['product', 'cart', 'purchase'])].copy()

In [ ]:
# Sort events within each session
funnel_events=funnel_events.sort_values(['session_id', 'sequence_number'])

In [ ]:
funnel_data=[]

In [ ]:
# Looping through each session
for session_id, session in funnel_events.groupby('session_id'):
    event_list=session['event_type'].to_list()

    # product view
    viewed_product='product' in event_list

    # cart after product
    added_cart=False

    if viewed_product:
        product_index=event_list.index('product')

        if 'cart' in event_list[product_index+1:]:
            added_cart=True

    # purchased after cart
    purchased=False

    if added_cart:
        cart_index=event_list.index('cart')

        if 'purchase' in event_list[cart_index+1:]:
            purchased=True

    # saving session results
    funnel_data.append({
        'session_id':session_id,
        'viewed_product':viewed_product,
        'added_cart':added_cart,
        'purchased':purchased
   })

In [ ]:
# Converting to dataframe
ordered_funnel=pd.DataFrame(funnel_data)

In [ ]:
ordered_funnel.iloc[:,1:]=(ordered_funnel.iloc[:,1:]).astype(int)

In [ ]:
# Funnel Conversion Rates
funnel_rates=ordered_funnel[['viewed_product', 'added_cart', 'purchased']].mean()
print(funnel_rates)

In [ ]:
# Conditional Funnel Conversions

# product to cart conversion
product_to_cart = (
    ordered_funnel['added_cart'].sum()
    /
    ordered_funnel['viewed_product'].sum()
)

print("\nProduct to Cart Conversion Rate:")
print(round(product_to_cart * 100, 2), "%")

In [ ]:
# cart to purchase conversion
cart_to_purchase = (
    ordered_funnel['purchased'].sum()
    /
    ordered_funnel['added_cart'].sum()
)

print("\nCart to Purchase Conversion Rate:")
print(round(cart_to_purchase * 100, 2), "%")

In [ ]:
# Drop-off Rates

product_to_cart_dropoff = 1 - product_to_cart
cart_to_purchase_dropoff = 1 - cart_to_purchase

print("\nProduct to Cart Drop-off Rate:")
print(round(product_to_cart_dropoff * 100, 2), "%")

print("\nCart to Purchase Drop-off Rate:")
print(round(cart_to_purchase_dropoff * 100, 2), "%")

In [ ]:
ordered_funnel[['viewed_product', 'added_cart', 'purchased']].mean().plot(kind='bar')
plt.title('Conversion Rates')
plt.show()

#### Purchase by Traffic Source

In [ ]:
purchase_by_traffic_source=events[events['event_type']=='purchase'].groupby('traffic_source')['session_id'].nunique()
print(purchase_by_traffic_source)
purchase_by_traffic_source.plot(kind='bar')
plt.title('Purchase Count by Traffic Source')
plt.xlabel('Traffic Source')
plt.ylabel('Purchase Count')
plt.show()

Email traffic demonstrates the highest session-to-purchase conversion rate and may warrant further investigation for future marketing investment.

### Conversion Rate by Traffic Source

In [ ]:
for source in (events['traffic_source']).unique():
    print(source)

In [ ]:
source_events=events.groupby('traffic_source').size()
source_events

In [ ]:
ordered_funnel.head()

In [ ]:
session_source=events.groupby('session_id')['traffic_source'].first().reset_index()

funnel_with_source=ordered_funnel.merge(session_source, on='session_id', how='left')

In [ ]:
funnel_with_source

In [ ]:
total_sessions=events.groupby('traffic_source')['session_id'].nunique()

source_funnel=funnel_with_source.groupby('traffic_source').agg(
    total=('session_id', 'count'),
    viewed=('viewed_product', 'sum'),
    carted=('added_cart', 'sum'),
    purchased=('purchased', 'sum')
)

source_funnel['view_rate']=source_funnel['viewed']/source_funnel['total']
source_funnel['cart_rate']=source_funnel['carted']/source_funnel['viewed']
source_funnel['purchase_rate']=source_funnel['purchased']/source_funnel['carted']
source_funnel['overall_conversion']=source_funnel['purchased']/source_funnel['total']

In [ ]:
source_funnel

In [ ]:
sources=source_funnel.index.tolist()
metrics=['view_rate', 'cart_rate', 'purchase_rate', 'overall_conversion']
labels=['View Rate', 'Cart Rate', 'Purchase Rate', 'Overall Conversion']

In [ ]:
x=np.arange(len(sources))
width=0.2

fig, ax=plt.subplots(figsize=(12, 6))

for i, (metric, label) in enumerate(zip(metrics, labels)):
    ax.bar(x+i*width, source_funnel[metric]*100, width, label=label)

ax.set_xticks(x+width*1.5)
ax.set_xticklabels(sources)
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Funnel Conversion Rate by Traffic Source')
ax.legend()
plt.tight_layout()
plt.show()

Conversion rates are remarkably consistent across all traffic sources, with overall session-to-purchase rates ranging narrowly between 26.5% and 26.9%. Facebook shows a marginally higher cart rate (63.8%) while Organic leads slightly in cart-to-purchase conversion (42.2%).

However, given the negligible differences, conversion rate alone does not differentiate traffic sources quality. Volume and acquisition cost would be more meaningful factors - Email drives the highest purchase volume (81,706) at comparable efficiency to other sources, suggesting it is the most scalable acquisition channel in absolute terms.

In [ ]:
orders=pd.read_csv('orders.csv')
print(orders.shape)
orders.head()

In [ ]:
orders.isnull().sum()

In [ ]:
orders.info()

In [ ]:
orders['created_at']=pd.to_datetime(orders['created_at'], format='mixed')
orders['shipped_at']=pd.to_datetime(orders['shipped_at'], format='mixed')
orders['delivered_at']=pd.to_datetime(orders['delivered_at'], format='mixed')
orders['returned_at']=pd.to_datetime(orders['returned_at'], format='mixed')

In [ ]:
orders['days_to_deliver']=(orders['delivered_at']-orders['created_at']).dt.days
print(orders['days_to_deliver'].mean())

On average, it took **3.5** days for the products to get delivered.

In [ ]:
print(round((orders['status']=='Cancelled').mean()*100,2))

**14.86%** of the total orders got cancelled.

In [ ]:
ord_items=pd.read_csv('order_items.csv')
print(ord_items.shape)
ord_items.head()

In [ ]:
ord_items.info()

In [ ]:
ord_items['created_at']=pd.to_datetime(ord_items['created_at'], format='mixed')
ord_items['shipped_at']=pd.to_datetime(ord_items['shipped_at'], format='mixed')
ord_items['delivered_at']=pd.to_datetime(ord_items['delivered_at'], format='mixed')
ord_items['returned_at']=pd.to_datetime(ord_items['returned_at'], format='mixed')

In [ ]:
ord_items.info()

All fields are now of the correct datatype.

In [ ]:
users=pd.read_csv('users.csv')
users.head()

In [ ]:
event_user=events.merge(users, left_on='user_id', right_on='id')
event_user.head()

In [ ]:
purchase_count_by_gender=event_user[event_user['event_type']=='purchase'].groupby('gender').count()
print(purchase_count_by_gender['id_x'])
purchase_count_by_gender['id_x'].plot(kind='pie')
plt.title('Purchase Count by Gender')
plt.xlabel('Gender')
plt.ylabel('Purchase Count')
plt.show()

**Females have made more purchases but the difference is negligible.**

In [ ]:
products=pd.read_csv('products.csv')
products.head()

### Revenue Analysis

In [ ]:
product_ord_items=products.merge(ord_items, left_on='id', right_on='product_id')
product_ord_items.head()

In [ ]:
product_ord_items=product_ord_items.rename(columns={'id_x':'product_master_id', 'id_y':'order_item_id'})

In [ ]:
top_categories=product_ord_items.groupby('category')['sale_price'].sum().sort_values(ascending=False)
top_categories.head(10)

In [ ]:
top_categories.head(10).plot(kind='barh')
plt.title('Revenue by Category')
plt.xlabel('Revenue')
plt.ylabel('Category')
plt.show()

### Brand Analysis

In [ ]:
top_brands=product_ord_items.groupby('brand')['sale_price'].sum().sort_values(ascending=False)
print(top_brands.head(10))
top_brands.head(10).plot(kind='bar')
plt.ylabel('Revenue')
plt.title('Revenue by Brand')
plt.show()

These brands contribute the largest share of revenue and warrant further investigation for promotional prioritization.

In [ ]:
product_ord_items['brand'].value_counts().head(10)

These are the top 10 brands in terms of purchase count. These are the most popular brands.

In [ ]:
print(product_ord_items.groupby('department')['sale_price'].sum())
product_ord_items.groupby('department')['sale_price'].sum().plot(kind='pie')
plt.ylabel('Revenue')
plt.title('Revenue by Department')
plt.show()

Although women have made more purchases, men's department has brought in more revenue. Prioritising marketing for men's department products could be a higher-revenue opportunity.

### Cohort Analysis

In [ ]:
# Getting each user's first month purchase
orders['order_month']=orders['created_at'].dt.to_period('M')
                       
first_purchase=(orders.groupby('user_id')['order_month'].min().reset_index())
first_purchase.columns=['user_id', 'cohort_month']

In [ ]:
# Merging cohort month back onto orders
orders_cohort=orders.merge(first_purchase, on='user_id')

In [ ]:
# Calculating months since first purchase
orders_cohort['months_since_first']=(orders_cohort['order_month']-orders_cohort['cohort_month']).apply(lambda x:x.n)

In [ ]:
# Counting distinct users per cohort period
cohort_data=orders_cohort.groupby(['cohort_month', 'months_since_first'])['user_id'].nunique().reset_index()

cohort_pivot=cohort_data.pivot(index='cohort_month', columns='months_since_first', values='user_id')

In [ ]:
# Converting to retention rate
cohort_retention=cohort_pivot.divide(cohort_pivot[0], axis=0)*100

In [ ]:
cohort_retention_filtered=cohort_retention[cohort_retention.index>=pd.Period('2022-01', 'M')]
cohort_retention_filtered=cohort_retention_filtered.iloc[:, :13]

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(cohort_retention_filtered, annot=True, fmt='.0f', cmap='YlGnBu', vmin=0, vmax=100)

plt.title('Monthly Cohort Retention Rate (%)')
plt.xlabel('Months Since First Purchase')
plt.ylabel('Cohort Month')
plt.tight_layout()
plt.show()

In [ ]:
# Exclude column 0 (that's always 100%) and young cohorts
stable_cohorts = cohort_retention[
    cohort_retention.index < pd.Period('2023-09', 'M')
]

# Drop column 0
retention_only = stable_cohorts.iloc[:, 1:]

print("Mean retention by month:")
print(retention_only.mean().round(1))

print("\nOverall mean retention (months 1-12):")
print(retention_only.stack().mean().round(1))

print("\nMin:", round(retention_only.stack().min(),2))
print("Max:", round(retention_only.stack().max(),2))

In [ ]:
# What % of users made more than 1 purchase?
repeat_buyers = orders.groupby('user_id')['order_id'].nunique()
print((repeat_buyers > 1).mean() * 100)  # % of users who are repeat buyers

In [ ]:
# Average number of purchases per repeat buyer
repeat_buyers = orders.groupby('user_id')['order_id'].nunique()
print(repeat_buyers[repeat_buyers > 1].mean().round(2))

# Distribution
print(repeat_buyers.value_counts().sort_index().head(10))

62.3% of customers made only one purchase, while 37.7% returned at least once. Among repeat buyers, the average order count was 2.5. The steepest drop-off occurs between the second and third purchase — 20,231 customers bought twice but only ~5,000 progressed to a third purchase. Combined with a flat monthly retention rate of ~2.8%, this suggests repeat purchases happen infrequently over long intervals rather than regularly. Re-engagement campaigns targeted at customers 3–6 months after their second purchase represent the highest-leverage retention opportunity.

# Key Findings

### Funnel
- 63.39% of product viewers added an item to cart
- 42.06% of cart sessions completed a purchase
- Overall product-view to purchase rate: 26.7%

### Traffic Source
- Email drove the highest purchase volume
- Conversion rates are consistent across all traffic sources

### Revenue
- Men's department generated higher total revenue ($5.74M) than Women's ($5.09M) despite women making slightly more purchases, suggesting higher average order value in men's products
- Cancellation rate stood at 14.86%

### Retention
- 37.7% of customers made more than one purchase
- Average repeat buyers placed 2.5 orders
- Steepest drop-off occurs between 2nd and 3rd purchase (20,231 vs ~5,000 customers)
- Mean monthly cohort retention rate averaged 12.8%, indicating purchases are infrequent but a meaningful repeat-buyer segment exists

# Limitations

- **Session-based funnel:** The funnel is constructed at session level, not user level. A single user abandoning and returning across multiple sessions is counted as separate funnel entries.

- **Traffic source attribution:** Attribution uses the source recorded at the time of the event. Multi-touch attribution (where a user arrives via multiple sources before purchasing) is not accounted for.

- **Cohort retention scope:** Cohort analysis measures repeat purchasing 
  behavior only. Browsing activity, wishlist additions, or other engagement 
  signals between purchases are not captured.

- **Causal inference:** Associations identified in this analysis (e.g. between traffic source and conversion, or department and revenue) are correlational. No causal claims are made without further experimental validation such as A/B testing.